# Taking apart an API you can see both sides of

**Module 3, Part 2 - APIs (companion to `16-apis.ipynb`)**

In `16-apis.ipynb` every API was a black box: we sent a request, got a
response, and guessed at what happened in between. This notebook uses an
API whose other side you can read as well.

**wx** shows eight weather models side by side. It answers terminals,
browsers and programs from the same URL, and its page
[Under the hood](https://wx.rejusamjohn.workers.dev/under-the-hood) explains every step it takes with your request.
We will poke it from the outside and check that what it says about itself
is true.

| Section | The idea |
|---|---|
| 1 | A request is just a URL |
| 2 | Status, body, and who is asking |
| 3 | Watching it go wrong |
| 4 | JSON to DataFrame |
| 5 | Check the provider |
| 6 | Parameters |
| 7 | The other side of the window |

## 0. Setup

Nothing here needs an account. wx was built for this course and runs on
free tiers, so treat it the way `16-apis.ipynb` asked you to treat Stack
Exchange: you are a guest.

> **Sharing a network?** wx allows each IP address 30 requests a minute. One
> run of this notebook makes about ten requests. If a cell comes back with
> `429`, that is section 7 happening to you: wait a minute and run it again.

[Weather data by Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0)

In [1]:
import json
import os
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests

BASE = os.environ.get("WX_BASE", "https://wx.rejusamjohn.workers.dev")
CACHE = Path("data") / "api-cache"

print("requests", requests.__version__)
print("pandas  ", pd.__version__)
print("talking to", BASE)

requests 2.34.2
pandas   2.3.3
talking to https://wx.rejusamjohn.workers.dev


### The safety net

Same idea as in `16-apis.ipynb`. Every JSON call below goes through
`fetch()`. It tries wx first. If the network is down, or wx is having a bad
day, it falls back to a copy saved in `data/api-cache/` and **tells you
loudly** that it has done so.

The cells that call `requests.get` directly have no safety net on purpose:
they are there to show you status codes and headers, and a saved copy has
neither.

In [7]:
def fetch(path, cache_name, params=None, timeout=15):
    """Get JSON from wx. Fall back to a saved copy, loudly, if that fails."""
    try:
        response = requests.get(BASE + path, params=params, timeout=timeout)
        response.raise_for_status()
        data = response.json()
        print("LIVE:", response.url[:95])
        return data
    except Exception as problem:
        snapshot = CACHE / cache_name
        if not snapshot.exists():
            raise
        print("=" * 66)
        print("!! LIVE CALL FAILED:", type(problem).__name__)
        print("!! Using the saved copy at", snapshot)
        print("!! The numbers below are NOT current.")
        print("=" * 66)
        return json.loads(snapshot.read_text())


print("cache folder exists:", CACHE.exists())

cache folder exists: True


If you downloaded this notebook from the wx site, `data/api-cache/` is empty and `fetch()` has nothing to fall back on yet. This cell saves the two copies it uses, once, while the site is up. In the course folder they are already there, so it just says so.

In [ ]:
SNAPSHOTS = {
    "wx-auckland.json": "/auckland?format=json&m",
    "wx-auckland-ecmwf-gfs.json": "/auckland?format=json&m&models=ecmwf,gfs&days=2",
}

def save_snapshots():
    """Save copies for fetch() to fall back on. Run once while the site is up."""
    CACHE.mkdir(parents=True, exist_ok=True)
    for name, path in SNAPSHOTS.items():
        target = CACHE / name
        if target.exists():
            print("already saved:", target)
            continue
        response = requests.get(BASE + path, timeout=15)
        response.raise_for_status()
        target.write_text(json.dumps(response.json(), indent=2))
        print("saved:", target)

save_snapshots()

---

## 1. A request is just a URL

Here is a question we will ask wx later: the Auckland forecast, for two
days, from two of the eight models. Before sending it, take it apart.

In [3]:
url = BASE + "/auckland?days=2&models=ecmwf,gfs"
parts = urlparse(url)

print("scheme (how to talk)    :", parts.scheme)
print("host   (which computer) :", parts.netloc)
print("path   (which place)    :", parts.path)
print("query  (which options)  :", parts.query)

scheme (how to talk)    : https
host   (which computer) : wx.rejusamjohn.workers.dev
path   (which place)    : /auckland
query  (which options)  : days=2&models=ecmwf,gfs


The same four parts as the ISS URL in `16-apis.ipynb`, with one addition:
the **query string**, everything after the `?`.

wx splits the job cleanly:

- The **path** is the place. `/auckland`, `/wellington`, `/-41.29,174.78`.
- The **query** is the options. `days=2`, `models=ecmwf,gfs`, each pair
  joined with `&`.

Change the path and you ask about somewhere else. Change the query and you
ask the same question about the same place in a different way.

In [4]:
assert parts.scheme == "https"
assert parts.path == "/auckland"
assert parts.query == "days=2&models=ecmwf,gfs"
print("Checks passed: place in the path, options in the query.")

Checks passed: place in the path, options in the query.


---

## 2. Status, body, and who is asking

In `16-apis.ipynb` the body of a URL was always the same shape. wx is less
predictable: it looks at **who is asking** before it decides what to send.

> **Predict first.** We ask for `/auckland` three times: once as plain `requests`, once pretending to be a web browser, and once adding `?format=json`. Same status each time? Same body?
>
> Put your answer in the chat before we run it.

In [5]:
askers = [
    ("plain requests", BASE + "/auckland", {}),
    ("pretend browser", BASE + "/auckland", {"User-Agent": "Mozilla/5.0"}),
    ("ask for JSON", BASE + "/auckland?format=json", {}),
]

content_types = []
statuses = []

for label, address, headers in askers:
    r = requests.get(address, headers=headers, timeout=15)
    content_types.append(r.headers["Content-Type"])
    statuses.append(r.status_code)
    print(label)
    print("  status      :", r.status_code)
    print("  content type:", r.headers["Content-Type"])
    print("  first 60    :", repr(r.text[:60]))
    print()

plain requests
  status      : 200
  content type: text/plain; charset=utf-8
  first 60    : '\x1bAuckland, New Zealand\x1b\x1b  Wed 16 Sep'

pretend browser
  status      : 200
  content type: text/html; charset=utf-8
  first 60    : '<!doctype html>\n<html lang="en-NZ">\n<head>\n<meta charset="ut'

ask for JSON
  status      : 200
  content type: application/json; charset=utf-8
  first 60    : '{"location":{"name":"Auckland","country":"New Zealand","lati'



Same URL, same status, three different bodies. This is **content
negotiation**: the server reads the request headers and picks a format.

- Every request carries a `User-Agent` header naming the program that sent
  it. `requests` sends `python-requests/<version>` unless you say otherwise.
- wx keeps a list of terminal tools (curl, wget, httpie, `python-requests`
  and a few more). If you are on the list you get **plain text**, the panel
  you would see in a terminal. The odd characters at the start are colour
  codes for that terminal.
- Anything else is treated as a browser and gets **HTML**.
- `format=json` overrides both, and you get **JSON** whoever you are.

So the status told you it worked, and only the `Content-Type` header told
you what you actually got. Check both before calling `.json()`.

In [ ]:
if 429 in statuses or 503 in statuses:
    print("429 or 503: this address has used its allowance, or the shared upstream budget is spent (see section 7). Wait a minute and re-run this cell.")
else:
    assert content_types[0].startswith("text/plain"), content_types[0]
    assert content_types[1].startswith("text/html"), content_types[1]
    assert content_types[2].startswith("application/json"), content_types[2]
    print("Checks passed:", [c.split(";")[0] for c in content_types])

---

## 3. Watching it go wrong

Two ways to ask wx a bad question: an option it does not know, and a place
that does not exist.

> **Predict first.** `/auckland?zzz` uses an option that does not exist, and `/zzqqxx` is not a place. Does `requests` raise an error for either? What status codes do you expect?
>
> Put your answer in the chat before we run it.

In [ ]:
bad_option = requests.get(BASE + "/auckland?zzz", timeout=15)
print("status:", bad_option.status_code)
print(bad_option.text)

no_place = requests.get(BASE + "/zzqqxx", timeout=15)
print("status:", no_place.status_code)
print(no_place.text)

**Neither raised.** Both are complete conversations with a "no" in them,
exactly like the `404` in `16-apis.ipynb`.

They are different kinds of "no", though:

- **`400 Bad Request`** means the request itself is malformed. wx is
  helpful about it: the body lists the options it does understand.
- **`404 Not Found`** means the request was fine, but there is nothing
  called that.

Both are `4xx`, so both are yours to fix, and retrying unchanged will never
help.

Now watch `.json()` meet that `400` body, which is plain text.

In [ ]:
try:
    bad_option.json()
except requests.exceptions.JSONDecodeError as problem:
    print("raised:", type(problem).__name__)

`JSONDecodeError`, the same failure as in `16-apis.ipynb`. We wrapped it in
`try`/`except` here so the notebook keeps running, but the lesson has not
changed: look at the status before you trust the body.

In [ ]:
if bad_option.status_code in (429, 503) or no_place.status_code in (429, 503):
    print("429 or 503: this address has used its allowance, or the shared upstream budget is spent (see section 7). Wait a minute and re-run this cell.")
else:
    assert bad_option.status_code == 400, bad_option.status_code
    assert no_place.status_code == 404, no_place.status_code
    print("Checks passed: 400 for a bad option, 404 for a missing place.")

---

## 4. JSON to DataFrame

`format=json` gets JSON. The extra `m` asks for degrees Celsius; it matters
in section 7.

In [8]:
forecast = fetch("/auckland?format=json&m", "wx-auckland.json")

print(sorted(forecast))

LIVE: https://wx.rejusamjohn.workers.dev/auckland?format=json&m
['attribution', 'daily', 'data_time', 'generated_at', 'hourly', 'location', 'models', 'stale', 'units']


In [9]:
print("location:", forecast["location"])
print("units   :", forecast["units"])
print("stale   :", forecast["stale"])
print("models  :", forecast["models"])

location: {'name': 'Auckland', 'country': 'New Zealand', 'latitude': -36.84853, 'longitude': 174.76349, 'timezone': 'Pacific/Auckland'}
units   : {'temperature': 'C', 'precipitation': 'mm'}
stale   : False
models  : ['ecmwf_ifs025', 'gfs_seamless', 'icon_seamless', 'ukmo_seamless', 'jma_seamless', 'gem_seamless', 'meteofrance_seamless', 'cma_grapes_global']


The model names in the JSON are longer than the ones you typed in the URL:
`ecmwf` is the short name for `ecmwf_ifs025`. The JSON uses the full
identifiers Open-Meteo uses.

`hourly` is a dictionary of lists, which is a shape pandas takes directly.
`time` becomes the index and each model becomes a column.

In [ ]:
hourly = pd.DataFrame(forecast["hourly"]["temperature"], index=pd.to_datetime(forecast["hourly"]["time"]))
hourly.index.name = "local time"
hourly.head()

The times are **local** Auckland time with no offset written on them. The
JSON tells you which zone they belong to in `location.timezone`, and in
section 5 we use it.

Now count the gaps.

In [ ]:
print("rows   :", len(hourly))
print()
print("missing hours per model:")
print(hourly.isna().sum())

Every model is asked for the same week, but not every model forecasts that
far ahead. Where a model's horizon ends, wx sends `null` in the JSON,
Python reads it as `None`, and pandas turns it into `NaN`. The gap is real:
that model has nothing to say about those hours.

`daily` is a list of dictionaries, one per day, which pandas also takes
directly.

In [ ]:
daily = pd.DataFrame(forecast["daily"])[["date", "consensus_max", "consensus_min", "spread_max"]]
daily

In [ ]:
assert list(hourly.columns) == forecast["models"]
assert len(hourly) == len(forecast["hourly"]["time"])
assert set(daily.columns) == {"date", "consensus_max", "consensus_min", "spread_max"}
print("Checks passed:", len(hourly), "hours,", len(hourly.columns), "models,", len(daily), "days.")

---

## 5. Check the provider

wx does not just pass the models along. For every hour it works out a
**median** (the consensus), a **25th percentile** and a **75th
percentile**. [Under the hood](https://wx.rejusamjohn.workers.dev/under-the-hood) says those use the same method as
NumPy's default.

That is a claim. We have the raw model values in the same response, so we
can test it rather than take it on trust.

First, find the row for the current hour. `generated_at` is in UTC, and the
hourly times are local, so convert before looking it up.

In [ ]:
tz = forecast["location"]["timezone"]
now_local = pd.Timestamp(forecast["generated_at"]).tz_convert(tz)
hour_key = now_local.strftime("%Y-%m-%dT%H:00")
i = forecast["hourly"]["time"].index(hour_key)

print("generated at (UTC):", forecast["generated_at"])
print("in", tz, ":", now_local)
print("row", i, "is", hour_key)

In [ ]:
values = [forecast["hourly"]["temperature"][m][i] for m in forecast["models"]]
values = [v for v in values if v is not None]

ours = {
    "median": float(np.percentile(values, 50)),
    "p25": float(np.percentile(values, 25)),
    "p75": float(np.percentile(values, 75)),
}
for key, value in ours.items():
    theirs = forecast["hourly"][key][i]
    print(f"{key:>6}: ours {value:6.2f}   wx {theirs:6.1f}")
    assert abs(value - theirs) <= 0.11, (key, value, theirs)
print("Checks passed: wx's statistics match NumPy's default percentile.")

**Why a tolerance of 0.11 and not an exact match?** wx works out the
statistics from the full-precision model values, and only then rounds
everything to one decimal place for the JSON. We computed ours from the
already rounded values. Each rounded value can be up to 0.05 away from the
original, so our answer can land up to about 0.1 away from wx's, and the
extra 0.01 allows for floating point.

Asking for an exact match would make the check fail for a reason that has
nothing to do with whether wx is right. Choosing the tolerance on purpose,
and being able to say why, is part of the check.

### A `200` is not the truth

Section 5 of `16-apis.ipynb` showed a `200` carrying out-of-date data with
nothing to warn you. wx tries to be honest about that.

In [ ]:
made = pd.Timestamp(forecast["generated_at"])
data = pd.Timestamp(forecast["data_time"])

print("stale       :", forecast["stale"])
print("generated_at:", forecast["generated_at"], " (when this response was made)")
print("data_time   :", forecast["data_time"], " (when wx got this forecast from Open-Meteo)")
print("age of data :", made - data)

- **`generated_at`** is when wx built this response.
- **`data_time`** is when wx fetched the forecast underneath it. If it is
  earlier, you were served a cached copy.
- **`stale`** is `true` when wx could not get a fresh copy and served an
  older one instead. It still answers `200`, and it tells you.

That is what `astros.json` in `16-apis.ipynb` did not do. A provider that
labels its own freshness saves you from having to find a second source to
date the data.

Now look at the models themselves for this hour.

> **Predict first.** Here are eight forecasts for the same place and the same hour. Which model is right?
>
> Put your answer in the chat before we run it.

In [ ]:
this_hour = hourly.iloc[i].dropna().sort_values()

print("temperature at", hour_key, "(C)")
print(this_hour.to_string())
print()
print("spread (warmest minus coolest): {:.1f} C".format(this_hour.max() - this_hour.min()))
print("wx max - min                  : {:.1f} C".format(forecast["hourly"]["max"][i] - forecast["hourly"]["min"][i]))

Nobody knows yet, and that is the point. Every one of those numbers
arrived with a `200` and valid JSON. **Different answers are not an
error**: they are the uncertainty in the forecast, made visible. A single
model on a weather app hides that; putting the models side by side shows
it.

So there are now three separate questions to ask of a response, and a
`200` only answers the first:

1. Did it work? (the status)
2. Is it current? (`data_time`, `stale`)
3. How sure is it? (the spread)

---

## 6. Parameters

The URL from section 1, now sent for real, as JSON.

In [ ]:
small = fetch("/auckland?format=json&m&models=ecmwf,gfs&days=2", "wx-auckland-ecmwf-gfs.json")

small_hourly = pd.DataFrame(small["hourly"]["temperature"], index=pd.to_datetime(small["hourly"]["time"]))

print("models :", small["models"])
print("columns:", list(small_hourly.columns))
print("hours  :", len(small_hourly), "  days:", len(small["daily"]))

`models` shrank from eight to two, and the hourly columns followed.

Look at the row counts, though. `days=2` did **not** shorten the JSON. In
wx, `days` sets how many days the terminal panel draws; the JSON always
carries the whole week. An API accepting a parameter does not mean every
format uses it, and the only way to find out is to look at what came
back.

### Let `requests` build the query string

Gluing options into a string works, but `16-apis.ipynb` recommended a
dictionary. Here is the same request built that way, before it is sent:

In [ ]:
options = {"format": "json", "models": "ecmwf,gfs", "days": 2}

prepared = requests.Request("GET", BASE + "/auckland?m", params=options).prepare()
print(prepared.url)

Two things to notice:

- The comma became `%2C`. That is URL encoding, and wx decodes it back to a
  comma.
- `m` stayed in the path string. It is a **flag**, an option with no value,
  and a dictionary wants a value for every key. Keeping flags in the path
  and everything else in `params` is the least surprising way to mix them.

In [ ]:
same = fetch("/auckland?m", "wx-auckland-ecmwf-gfs.json", params=options)

assert same["models"] == small["models"], (same["models"], small["models"])
assert len(small["models"]) == 2
print("Checks passed: string and dictionary ask the same question.")

### A bad value

The option name is right this time, but the value is not.

In [ ]:
nope = requests.get(BASE + "/auckland?models=nope", timeout=15)

print("status:", nope.status_code)
print(nope.text)

if nope.status_code in (429, 503):
    print("429 or 503: this address has used its allowance, or the shared upstream budget is spent (see section 7). Wait a minute and re-run this cell.")
else:
    assert nope.status_code == 400, nope.status_code
    print("Check passed: a bad value is a 400, and the body says which value.")

---

## 7. The other side of the window

`16-apis.ipynb` called an API a service window on the side of a building.
Here is what happens behind wx's window every time you send a request. The
full version, with the reasons, is on [Under the hood](https://wx.rejusamjohn.workers.dev/under-the-hood).

1. **Method.** Only `GET` and `HEAD` are answered. Anything else gets `405`.
2. **Options.** The path is the place and the query holds the options. An
   unknown option or a bad value gets `400`, and the body lists the valid
   options. (You saw this in sections 3 and 6.)
3. **Your allowance.** Each IP address may make **30 requests a minute**.
   After that: `429 Too Many Requests`, with a `Retry-After` header.
4. **The place.** A place name goes to a geocoding service once, and the
   answer is kept for **30 days**. A name that is not found is remembered
   for **6 hours** (`404`), so a typo costs the shared budget once, not on
   every attempt.
5. **The forecast cache.** Coordinates are rounded to 0.1 degrees (about
   11 km), so nearby requests share one copy. A copy is **fresh for 30
   minutes** and **kept for 6 hours**.
6. **The shared budget.** Only a cache miss calls Open-Meteo, the service
   the forecasts come from, and it has to fit two budgets: **3 uncached
   lookups a minute per IP address**, and **about 5 a minute per Cloudflare
   location**.
7. **When upstream says no.** If a budget is spent or Open-Meteo fails, wx
   serves the copy it kept and sets `"stale": true`. With no copy to fall
   back on, the answer is `503` for a spent budget or `502` for a failed
   call.
8. **The answer.** The same data becomes a terminal panel, a web page or
   JSON, and every data response carries an `X-Data-Source` header.

Most of those decisions leave a trace in the **response headers**.

In [ ]:
r = requests.get(BASE + "/auckland?format=json&m", timeout=15)

if r.status_code in (429, 503):
    print("429 or 503: this address has used its allowance, or the shared upstream budget is spent (see section 7). Wait a minute and re-run this cell.")
else:
    print("status                     :", r.status_code)
    for name in ["Cache-Control", "X-Data-Source", "Access-Control-Allow-Origin", "Vary"]:
        print(f"{name:<27}: {r.headers.get(name)}")
    print("elapsed                    :", r.elapsed.total_seconds(), "seconds")

What each header is telling a client:

- **`Cache-Control: public, max-age=600`**: anyone along the way, your
  browser or a shared cache, may reuse this answer for 600 seconds without
  asking again. Drop the `m` and it says `private`, because without a unit
  wx picks one based on where you are, and that answer should not be shared
  with other people.
- **`X-Data-Source`**: where the numbers came from and under which licence.
  A program that never shows the body to a person can still pass the
  attribution on.
- **`Access-Control-Allow-Origin: *`**: JavaScript running on any website
  may read this response. Without it, a browser would fetch the data and
  then refuse to hand it to the page.
- **`Vary: User-Agent, Accept`**: the answer depends on who is asking,
  which is section 2 in one line. A cache must not hand a browser the
  terminal version.
- **elapsed**: run the cell twice. The time depends on the network and on
  whether wx had to call Open-Meteo, so do not read much into one number.

> **Predict first.** Thirty students each ask wx for Auckland. Then thirty students each ask for a different town. Which of those can end in `503` for most of the class, and why?
>
> Put your answer in the chat before we run it.

Work it through with the eight steps.

**Thirty requests for Auckland.** The first one looks up the place and
fetches the forecast. The other twenty-nine find both in the cache. Two
upstream calls in total, and everyone gets a `200`.

**Thirty different towns.** Every town is a cache miss twice over: one
geocoding lookup and one forecast call, so about sixty upstream calls.
The budget in step 6 is about 5 a minute per Cloudflare location, and if
the whole class is on one wifi network it is also one IP address with 3
uncached lookups a minute. One or two students get through. For everyone
else the budget is spent, there is no cached copy of their town to fall
back on, and the answer is `503`.

Nothing is broken in the second case. The limits are doing their job:
protecting a free service that everyone else in the world is sharing too.
It is the server-side view of the Stack Exchange quota question in
`16-apis.ipynb`.

### Seeing a real `429`

The cell below asks for Auckland 35 times in a row. It should get a `429`
before the end, and then **every** request from this address fails for up to a
minute, including your neighbours' if you share wifi. So it is off by
default.

In [ ]:
RUN_429_DEMO = False   # instructor only: spends this address's allowance for a minute

if RUN_429_DEMO:
    for attempt in range(1, 36):
        r = requests.get(BASE + "/auckland?format=json&m", timeout=15)
        if r.status_code == 429:
            print("attempt", attempt, "->", r.status_code, "Retry-After:", r.headers.get("Retry-After"))
            break
    else:
        print("no 429 within 35 requests")
else:
    print("Skipped. Set RUN_429_DEMO = True to see a real 429 (instructor only).")

---

## What to take away

1. **A provider protects a shared budget with limits and caches.** Your
   allowance, the place cache, the forecast cache and the upstream budget
   all exist because a free service has to survive everyone using it at
   once.
2. **The headers tell you what it decided.** `Content-Type` says what you
   got, `Cache-Control` says how long you may keep it, `Retry-After` says
   when to come back.
3. **A `200` can be stale, and wx says so.** `stale` and `data_time` answer
   the freshness question from `16-apis.ipynb` inside the response itself.
   When an API does not, you have to ask it yourself.

---

*Data Science & AI - Module 3 Part 2, companion to `16-apis.ipynb`. Saved
wx responses live in `data/api-cache/`. How wx works:
[Under the hood](https://wx.rejusamjohn.workers.dev/under-the-hood).*

[Weather data by Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0)